# TTS bootcamp – run the baseline

1. Runtime → Change runtime type → **T4 GPU**.
2. Accept the model terms at https://huggingface.co/ai4bharat/IndicF5 and add a Colab secret named `HF_TOKEN` (key icon, left sidebar).
3. Put the unzipped kit in Google Drive (or upload it) and set `KIT` below.
4. Run all. If a cell says the length fix was just applied, choose Runtime → **Restart session** and run all again (happens once).

In [ ]:
KIT = "/content/drive/MyDrive/tts_bootcamp_kit"   # folder that contains metrics.py
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:], r.stderr[-3000:] if r.returncode else "")
    if r.returncode:
        raise RuntimeError(f"command failed: {cmd}")

os.chdir(KIT)
sys.path.insert(0, KIT)
sh("pip install -q -r baseline/requirements.txt")

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception as e:
    print("No HF_TOKEN secret found, falling back to interactive login:", e)
    login()

## Synthesize one item

In [ ]:
import csv
from IPython.display import Audio, display
from baseline.tts import synthesize, save_wav

items = list(csv.DictReader(open("data/val_items.csv", encoding="utf-8")))
prompts = {r["prompt_id"]: r for r in csv.DictReader(open("data/prompts.csv", encoding="utf-8"))}
it = items[0]
p = prompts[it["prompt_id"]]
audio = synthesize(it["text"], "data/" + p["file"], p["transcript"], seed=0)
save_wav("demo.wav", audio)
print(it["text"])
display(Audio("data/" + p["file"]), Audio("demo.wav"))

## Reproduce the baseline on all 24 validation items (and time it)

Writes `out/baseline_val/` and `out/baseline_val_timing.json`. That timing file is your baseline time for the Task 4 speed-up; rerun it in the same session as your fast system.

In [ ]:
sh("python baseline/run_baseline.py --items data/val_items.csv --out out/baseline_val --seed 0")
print(json.load(open("out/baseline_val_timing.json"))["total_synth_s"], "seconds")

## Score with metrics.py (Task 1 uses the provided `data/baseline_val/`)

In [ ]:
sh("python metrics.py data/baseline_val --json out/baseline_val_scores.json --csv out/baseline_val_scores.csv")
print(open("out/baseline_val_scores.json").read())